# Mitra Classifier — End-to-End Classification with Your Own Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-classifier-pipeline/blob/main/tutorials/mitra_classifier_colab.ipynb)

This standalone tutorial uses the **Mitra Classifier** checkpoint distributed through the DIMER Model Repository. It does **not** use DIMER Workbench, DIMER APIs, or DIMER validator/fine-tuner workers.

You can upload the ZIP downloaded from DIMER or use the exact pinned upstream checkpoint as a fallback, then bring your own CSV, evaluate pretrained Mitra, optionally fine-tune on a GPU, classify new rows, and export the resulting predictor.

> **Data handling:** your dataset is processed in your Google Colab runtime, not by DIMER. Do not upload confidential, sensitive, or restricted data unless its use in that environment is permitted.

## 1. Install the runtime

The `mitra` extra is required; plain `autogluon.tabular` does not include all Mitra runtime dependencies. PyTorch is left to the Colab runtime so its CUDA build stays compatible with the selected accelerator.

In [ ]:
%pip install -q "autogluon.tabular[mitra]==1.5.0"

## 2. Acquire and verify the checkpoint

DIMER hosts `model.safetensors`. This notebook retrieves the matching `config.json` from the exact upstream revision associated with this release. Both routes are SHA-256 verified, installed into an isolated local Hugging Face cache, and then used offline so AutoGluon cannot silently resolve a newer `main`.

In [ ]:
import hashlib, json, os, random, shutil, urllib.request, zipfile
from pathlib import Path

HF_HOME = Path("/content/mitra-hf")
os.environ["HF_HOME"] = str(HF_HOME)

MODEL_ID = "autogluon/mitra-classifier"
PINNED_REVISION = "c425e9fa0910a6be1c494321792e7ba2a1367b1a"
EXPECTED_WEIGHTS_SHA256 = "e06a055e91a3baeffc37f9cf634d9e69a27d904b6686131dc3b702f9c0126b19"
EXPECTED_CONFIG_SHA256 = "2c96c24dd25f64e92753f6f2ba00cc7833b9923459403dcd8504e8700c0995df"
MODEL_DIR = Path("/content/mitra-model"); MODEL_DIR.mkdir(exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

def download_pinned(name, dest):
    url = f"https://huggingface.co/{MODEL_ID}/resolve/{PINNED_REVISION}/{name}?download=true"
    print(f"Retrieving pinned {name}...")
    with urllib.request.urlopen(url) as r, open(dest, "wb") as f: shutil.copyfileobj(r, f)

def verify(path, expected, label):
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f"{label} checksum mismatch.\nExpected: {expected}\nActual:   {actual}")
    print(f"✓ {label} verified: {actual[:12]}…")

def weights_from_upload(dest):
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1: raise RuntimeError("Upload exactly one DIMER ZIP or model.safetensors.")
    p = Path("/content") / next(iter(uploaded))
    if p.suffix.lower() == ".safetensors":
        shutil.copy2(p, dest); return
    if p.suffix.lower() != ".zip": raise ValueError("Expected a DIMER ZIP or model.safetensors.")
    with zipfile.ZipFile(p) as z:
        matches = [i for i in z.infolist() if not i.is_dir() and Path(i.filename).name == "model.safetensors"]
        if len(matches) != 1: raise RuntimeError(f"Expected one model.safetensors; found {len(matches)}.")
        with z.open(matches[0]) as src, open(dest, "wb") as dst: shutil.copyfileobj(src, dst)

def install_offline_snapshot(weights, config):
    commit = sha256_file(weights)[:40]
    repo = HF_HOME / "hub" / ("models--" + MODEL_ID.replace("/", "--"))
    snap = repo / "snapshots" / commit; refs = repo / "refs"
    snap.mkdir(parents=True, exist_ok=True); refs.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weights, snap / "model.safetensors")
    shutil.copy2(config, snap / "config.json")
    (refs / "main").write_text(commit)
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    return snap

MODEL_SOURCE = "Pinned upstream"  # @param ["DIMER ZIP", "Pinned upstream"]
weights_path = MODEL_DIR / "model.safetensors"
config_path = MODEL_DIR / "config.json"

if MODEL_SOURCE == "DIMER ZIP":
    weights_from_upload(weights_path)
    download_pinned("config.json", config_path)
else:
    download_pinned("model.safetensors", weights_path)
    download_pinned("config.json", config_path)

verify(weights_path, EXPECTED_WEIGHTS_SHA256, "model.safetensors")
verify(config_path, EXPECTED_CONFIG_SHA256, "config.json")
print(json.dumps(json.loads(config_path.read_text()), indent=2))
print(f"✓ Offline snapshot: {install_offline_snapshot(weights_path, config_path)}")

## 3. Bring your own data

Upload one CSV where each row is an observation and one column is the categorical target. The built-in demo lets you verify the notebook before using your own data.

Mitra Classifier supports **2–10 classes**, up to **500 features**, and at most **10,000 training rows**. The checks below are notebook-side educational checks, not DIMER Workbench validation.

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

DATA_SOURCE = "Built-in demo"  # @param ["Built-in demo", "Upload CSV"]
TARGET_COLUMN = "target"       # @param {type:"string"}
DROP_COLUMNS = ""              # @param {type:"string"}
VALIDATION_SPLIT = 0.20        # @param {type:"number"}
SEED = 42                      # @param {type:"integer"}

if DATA_SOURCE == "Built-in demo":
    data = load_breast_cancer(as_frame=True).frame.copy()
    TARGET_COLUMN = "target"
else:
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".csv")]
    if len(names) != 1: raise RuntimeError("Upload exactly one CSV.")
    data = pd.read_csv(Path("/content") / names[0])

drop_columns = [c.strip() for c in DROP_COLUMNS.split(",") if c.strip() and c.strip() != TARGET_COLUMN]
if data.columns.duplicated().any(): raise ValueError("Duplicate column names are not supported.")
if TARGET_COLUMN not in data.columns: raise ValueError(f"Target column {TARGET_COLUMN!r} not found.")

clean = data.drop(columns=[c for c in drop_columns if c in data.columns], errors="ignore").dropna(subset=[TARGET_COLUMN]).copy()
features = [c for c in clean.columns if c != TARGET_COLUMN]
counts = clean[TARGET_COLUMN].value_counts()
n_classes = len(counts)

errors = []
if len(clean) < 50: errors.append("Use at least 50 labelled rows.")
if not features: errors.append("No feature columns remain.")
if len(features) > 500: errors.append(f"{len(features)} features exceed Mitra's 500-feature limit.")
if not 2 <= n_classes <= 10: errors.append(f"Target has {n_classes} classes; Mitra requires 2–10.")
if len(counts) and counts.min() < 2: errors.append(f"Every class needs at least 2 rows; counts={counts.to_dict()}.")
if errors: raise ValueError("Dataset is not ready:\n- " + "\n- ".join(errors))
if not 0.05 <= VALIDATION_SPLIT <= 0.40: raise ValueError("VALIDATION_SPLIT must be 0.05–0.40.")

display(pd.DataFrame({"Item":["Usable rows","Features","Target","Classes"],"Value":[len(clean),len(features),TARGET_COLUMN,n_classes]}))
display(counts.rename("rows").to_frame())
if len(clean) > 10_000: print("⚠ Dataset exceeds 10,000 rows; the training split will be class-preserving capped.")
near_unique = [c for c in features if clean[c].nunique(dropna=False) / len(clean) > 0.98]
if near_unique: print("⚠ Nearly unique/identifier-like columns:", near_unique[:10])

train_data, holdout_data = train_test_split(clean, test_size=VALIDATION_SPLIT, random_state=SEED, stratify=clean[TARGET_COLUMN])

def stratified_cap(df, ceiling=10_000):
    if len(df) <= ceiling: return df.reset_index(drop=True)
    rng = np.random.RandomState(SEED); keep = []
    for cls in df[TARGET_COLUMN].drop_duplicates():
        keep.append(rng.choice(df.index[df[TARGET_COLUMN] == cls].to_numpy()))
    remaining = np.array([i for i in df.index if i not in set(keep)])
    keep.extend(rng.choice(remaining, size=ceiling-len(keep), replace=False))
    return df.loc[keep].sample(frac=1, random_state=SEED).reset_index(drop=True)

train_data = stratified_cap(train_data)
FEATURE_COLUMNS = [c for c in train_data.columns if c != TARGET_COLUMN]
NUM_CLASSES = train_data[TARGET_COLUMN].nunique()
PROBLEM_TYPE = "binary" if NUM_CLASSES == 2 else "multiclass"
print(f"✓ train={len(train_data):,}, holdout={len(holdout_data):,}, features={len(FEATURE_COLUMNS)}, type={PROBLEM_TYPE}")

## 4. Evaluate pretrained Mitra

With `fine_tune=False`, Mitra uses the labelled training table as context but does not update its pretrained weights. This is the baseline to compare against fine-tuning.

In [ ]:
import torch
from autogluon.tabular import TabularPredictor

EVAL_METRIC = "accuracy"    # @param ["accuracy", "balanced_accuracy", "log_loss", "f1_macro", "mcc"]
BASELINE_TIME_LIMIT = 300   # @param {type:"integer"}

CUDA_AVAILABLE = torch.cuda.is_available()
print("CUDA available:", CUDA_AVAILABLE, torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "")

def seed_everything():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def fit_mitra(fine_tune, path, time_limit, fine_tune_steps=0):
    seed_everything()
    hp = {"fine_tune": fine_tune, "seed": SEED}
    if EVAL_METRIC in {"accuracy", "log_loss"}: hp["metric"] = EVAL_METRIC
    if fine_tune and fine_tune_steps > 0: hp["fine_tune_steps"] = fine_tune_steps
    predictor = TabularPredictor(label=TARGET_COLUMN, problem_type=PROBLEM_TYPE, eval_metric=EVAL_METRIC, path=path, verbosity=2)
    predictor.fit(train_data, hyperparameters={"MITRA": hp}, fit_weighted_ensemble=False, time_limit=time_limit)
    if not any("mitra" in n.lower() for n in predictor.model_names()):
        raise RuntimeError(f"Expected Mitra; AutoGluon trained {predictor.model_names()}.")
    return predictor

def evaluate(predictor):
    raw = predictor.evaluate(holdout_data, auxiliary_metrics=True, silent=True)
    return {k: float(-v if "log_loss" in k else v) for k,v in raw.items()}

baseline_predictor = fit_mitra(False, "/content/mitra-baseline", BASELINE_TIME_LIMIT)
baseline_metrics = evaluate(baseline_predictor)
display(pd.Series(baseline_metrics, name="Pretrained").to_frame())

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
pred = baseline_predictor.predict(holdout_data.drop(columns=[TARGET_COLUMN]))
ConfusionMatrixDisplay.from_predictions(holdout_data[TARGET_COLUMN], pred)
plt.title("Pretrained Mitra — holdout confusion matrix"); plt.show()

## 5. Optional fine-tuning

Fine-tuning updates Mitra's pretrained weights and requires a GPU in this notebook. It is an experiment, not an automatic upgrade: compare its holdout metrics against the pretrained baseline.

In [ ]:
RUN_FINE_TUNING = False      # @param {type:"boolean"}
FINE_TUNE_STEPS = 0          # @param {type:"integer"}
FINE_TUNE_TIME_LIMIT = 600   # @param {type:"integer"}

finetuned_predictor = finetuned_metrics = None
if RUN_FINE_TUNING:
    if not CUDA_AVAILABLE:
        raise RuntimeError("Fine-tuning requires a GPU. Choose Runtime → Change runtime type → GPU and rerun.")
    finetuned_predictor = fit_mitra(True, "/content/mitra-finetuned", FINE_TUNE_TIME_LIMIT, FINE_TUNE_STEPS)
    finetuned_metrics = evaluate(finetuned_predictor)
    keys = sorted(set(baseline_metrics) & set(finetuned_metrics))
    display(pd.DataFrame({"Pretrained":[baseline_metrics[k] for k in keys],"Fine-tuned":[finetuned_metrics[k] for k in keys]}, index=keys))
else:
    print("Fine-tuning skipped. Set RUN_FINE_TUNING=True on a GPU to run it.")

## 6. Classify new rows and export results

Upload a CSV with the same feature columns as the training data. The target is optional. Inference uses the fine-tuned predictor when available, otherwise the pretrained predictor.

The reusable export is an **AutoGluon predictor directory packaged as ZIP**, not a replacement `model.safetensors`. Reload it later with `TabularPredictor.load(path)`.

In [ ]:
RUN_NEW_DATA_INFERENCE = False  # @param {type:"boolean"}
DOWNLOAD_EXPORTS = False         # @param {type:"boolean"}

selected = finetuned_predictor or baseline_predictor
selected_mode = "fine-tuned" if finetuned_predictor is not None else "pretrained"
predictions_path = None

if RUN_NEW_DATA_INFERENCE:
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".csv")]
    if len(names) != 1: raise RuntimeError("Upload exactly one CSV.")
    new_data = pd.read_csv(Path("/content") / names[0])
    new_data = new_data.drop(columns=[TARGET_COLUMN] if TARGET_COLUMN in new_data.columns else [], errors="ignore")
    new_data = new_data.drop(columns=[c for c in drop_columns if c in new_data.columns], errors="ignore")
    missing = [c for c in FEATURE_COLUMNS if c not in new_data.columns]
    if missing: raise ValueError(f"Missing required features: {missing}")
    model_input = new_data[FEATURE_COLUMNS]
    out = new_data.copy()
    out["predicted_class"] = selected.predict(model_input).to_numpy()
    proba = selected.predict_proba(model_input)
    if isinstance(proba, pd.DataFrame):
        for cls in proba.columns: out[f"probability_{cls}"] = proba[cls].to_numpy()
    predictions_path = Path("/content/mitra_predictions.csv")
    out.to_csv(predictions_path, index=False)
    display(out.head()); print(f"✓ Saved {len(out):,} predictions.")

if DOWNLOAD_EXPORTS:
    from google.colab import files
    metrics = finetuned_metrics if finetuned_metrics is not None else baseline_metrics
    meta = {
        "base_model": MODEL_ID, "base_model_revision": PINNED_REVISION,
        "weights_sha256": EXPECTED_WEIGHTS_SHA256, "config_sha256": EXPECTED_CONFIG_SHA256,
        "model_source": MODEL_SOURCE, "mode": selected_mode, "target_column": TARGET_COLUMN,
        "problem_type": PROBLEM_TYPE, "num_classes": int(NUM_CLASSES),
        "feature_columns": FEATURE_COLUMNS, "seed": SEED, "eval_metric": EVAL_METRIC,
        "holdout_metrics": metrics,
    }
    meta_path = Path("/content/mitra_run_metadata.json")
    meta_path.write_text(json.dumps(meta, indent=2))
    archive = Path(shutil.make_archive(f"/content/mitra-{selected_mode}-predictor", "zip", root_dir=Path(selected.path)))
    files.download(str(archive)); files.download(str(meta_path))
    if predictions_path: files.download(str(predictions_path))
else:
    print("Set RUN_NEW_DATA_INFERENCE and/or DOWNLOAD_EXPORTS when you are ready.")

## What you just did

You verified a specific pretrained checkpoint, supplied labelled tabular data, evaluated Mitra on unseen rows, optionally fine-tuned it, and used the selected predictor for new-row classification.

Before deployment, repeat evaluation across sensible splits/seeds, choose metrics appropriate to class balance and decision cost, inspect features for leakage and identifiers, confirm data licensing/governance, and perform domain validation. For formal provenance and intended-use limits, refer to the Mitra Classifier model card distributed through DIMER.